In [1]:
# Imports 
import os
import re
import pickle

In [2]:
# Get absorption and emission file paths
def get_absorption_emission_files(directory):
    abs_files, ems_files = [], []
    for fp in os.listdir(directory):
        match = re.match(r'^(.*?)\(.*\)(.*)$', fp)
        if match:
            compound_solvent, file_type = match.groups()
            if file_type == '.abs.txt':
                abs_files.append((compound_solvent, fp))
            elif file_type == '.ems.txt':
                ems_files.append((compound_solvent, fp))
    return abs_files, ems_files

# Read in the text file and return wavelength and intensity
def read_txt_file(txt_file_path):
    wavelengths, intensities = [], []
    with open(txt_file_path, 'r') as file:
        lines = file.readlines()
        for i, line in enumerate(lines):
            if i > 0:
                line_split = line.split()
                if len(line_split) > 0:
                    wavelengths.append(float(line_split[0]))
                    intensities.append(float(line_split[1]))
    return wavelengths, intensities

# Find corresponding files for absorption and emission
def match_absorption_emission(directory, abs_files, ems_files):
    ems_dict = dict(ems_files)
    combined = {
        key: {'absorption': val1, 'emission': ems_dict.get(key)}
        for key, val1 in abs_files if key in ems_dict}
    wavelengths_and_intensities = dict()
    for compound_key in combined.keys():
        abs_wavelength, abs_intensity = read_txt_file(os.path.join(directory, combined[compound_key]['absorption']))
        ems_wavelength, ems_intensity = read_txt_file(os.path.join(directory, combined[compound_key]['emission']))
        compound_dict = dict()
        compound_dict['absorption'] = {'wavelength':abs_wavelength, 'intensity':abs_intensity}
        compound_dict['emission'] = {'wavelength':ems_wavelength, 'intensity':ems_intensity}
        wavelengths_and_intensities[compound_key] = compound_dict
    return wavelengths_and_intensities


In [3]:
# Compile data from entire natural chlorophylls database
database_path = './Natural Chlorophylls/'
abs_files, ems_files = get_absorption_emission_files(database_path)
spectra = match_absorption_emission(database_path, abs_files, ems_files)

In [18]:
# Sort by solvent
def get_molecules_from_solvent(spectral_db, solvent):
    # Get list of compounds
    compound_list = list(spectral_db.keys())
    # Compile spectra from specific solvent
    spectra = dict()
    for comp in compound_list:
        comp_solvent = comp.split(',')[-1]
        if comp_solvent == solvent:
            cmpd_name = ",".join(comp.split(',')[:-1])
            spectra[cmpd_name] = spectral_db[comp]
    return spectra


In [19]:
# Get the Et20 solvent molecules
et20_spectra = get_molecules_from_solvent(spectra, ' Et2O ')

In [17]:
print(et20_spectra.keys())

dict_keys(['CHL070_Chl b2', 'CHL215_Chl c', 'CHL205_BChl f [E,M]', 'CHL014_Chl b', 'CHL249_ProtoPhe a', 'CHL273_BPhe a', 'CHL047_Phe b', 'CHL021_Chl d', 'CHL064_Chl a2', 'CHL006_Chl a', 'CHL030_Ch f', 'CHL197_BChl d [E,E]+[P,M]', 'CHL052_Phe d', 'CHL202_BChl e [E,E]+[P,E]', 'CHL231_Phe c'])


In [20]:
# Save this as a pickle file
pickle.dump(et20_spectra, open('./et20_spectra.pkl', 'wb'))